# 17 · Phase C: test-set evaluation under a CU cap (L4, one session)

**L4 runtime, bf16** (as in training). Estimate about **9.45 CU** at 2.2 CU/h for every block (about 4.3 h). Type the Colab balance into `CU_BALANCE_START` (cell 2) before Run all; `CU_CAP` is 9.7 (Dhanush, 25 Nov; hard limit 11).

**Inputs, all checked before anything runs:**
- the six test-set tars of notebook 16 (`runs/cache/tests/`), each copied and checked member by member;
- the P6 image tar and the notebook 14 evidence bundles (CC);
- the checkpoints: unified final configuration seeds 42, 123, 2026; the detection-protocol model, seed 42;
- the cached CLIP probes of notebook 09 (`runs/cp4/probes/`; probe rows are skipped if missing);
- `eval_subsets.json` (P5, P6).

**Resuming.** `PHASE_C_BALANCE_START` (23.43, fixed) is the balance when Phase C first started. Type the balance Colab shows now into `CU_BALANCE_START`; this session's cap is `CU_CAP` minus what earlier Phase C sessions spent (23.43 − `CU_BALANCE_START`). The pre-flight prints it and refuses when nothing is left.

**A block that raises** (an out-of-memory error, a bad file, a mask of the wrong shape) is logged as `failed` with its traceback and minutes, nothing is saved for it, and the next block runs; cell 10 (result file, compute record, release) always runs.

**Images are prefetched**: decoding, harmonizing and perturbing run in 6 background threads, in order; each block logs its images/s.

**Blocks, in priority order.** Before each block, if the CU used so far (2.2 CU/h × wall time) plus the block's estimate exceeds `CU_CAP`, the session stops cleanly: finished blocks are saved, the rest are logged as skipped, and the notebook goes straight to the compute record and the release.

| Block | What | Estimate (CU) |
|---|---|---|
| `s42_p1`, `s42_p2`, `s42_p3` | unified seed 42: P1 raw + harmonized (with the unified probe), P2, P3 BR-Gen | 0.9, 0.35, 0.25 |
| `det_p1` | detection model + its probe: P1 raw + harmonized | 0.9 |
| `s123_*`, `s2026_*` | the same three for seeds 123 and 2026 | 1.5 each |
| `cc` | counterfactual consistency of every E region of P6 (seed 42) | 0.4 |
| `p3b`, `p4` | SID-Set validation (unified); Chameleon (both models) | 0.55, 0.2 |
| `p5_det`, `p5_uni` | P5: detection on the GenImage part, unified on the DiffSeg30k part, 13 conditions | 1.7, 0.4 |
| `timings` | images/s, tiles/s, GPU memory, parameters | 0.2 |

A dropped session resumes: a block whose result file is on Drive is reused when its inputs are unchanged (the code under `src/` and `configs/` and this notebook, the checkpoint, probe and tar hashes, the subsets, the P5 levels and the CC rule); otherwise it runs again. The code part is the git hash of `src/`, `configs/` and this notebook, so a docs-only merge in between keeps finished blocks.

**Results** (send to Claude): `Paper 1/runs/phase_c/phase_c_seed42.json` and `compute_phase_c.json`, plus the balance after the run.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings (type CU_BALANCE_START and CU_CAP first)
import csv, datetime, json, logging, shutil, tempfile, time
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
import numpy as np, torch
from eviground.baselines import clip_probe
from eviground.config import load_config
from eviground.data import harmonize, testcache
from eviground.env import get_paths, session_info
from eviground.eval import cp4, p6cache, perturb, phasec, runner, subsets, testsets
from eviground.explain import cc, evidence
from eviground.utils import session

SEED, SEEDS = 42, [42, 123, 2026]
CU_BALANCE_START = None    # REQUIRED: the balance Colab shows now
CU_CAP = 9.7               # Dhanush, 25 Nov: hard limit 11; the largest block (~1.7) cannot pass it
PHASE_C_BALANCE_START = 23.43   # fixed (Dhanush, 25 Nov): the balance when Phase C first started; a resumed
                                # session's cap is CU_CAP minus what earlier Phase C sessions spent
PREFETCH = {'workers': 6, 'depth': 64}   # background decode / harmonize / perturb, order kept
RATE = session.CU_RATES['L4']
EST = {'s42_p1': 0.9, 's42_p2': 0.35, 's42_p3': 0.25, 'det_p1': 0.9,
       's123_p1': 0.9, 's123_p2': 0.35, 's123_p3': 0.25, 's2026_p1': 0.9, 's2026_p2': 0.35, 's2026_p3': 0.25,
       'cc': 0.4, 'p3b': 0.55, 'p4': 0.2, 'p5_det': 1.7, 'p5_uni': 0.4, 'timings': 0.2}
SETUP_CU = 0.6             # staging the tars and loading the models (estimate)
EST_CU = round(sum(EST.values()) + SETUP_CU, 2)
T0 = time.time()
info = session_info(cu_per_hour=RATE)
paths = get_paths()
if DRY:
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_phasec_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    CU_BALANCE_START, CU_CAP, PHASE_C_BALANCE_START = 50.0, 40.0, 50.0
    EST['dry_fail'] = 0.0      # a block that raises: logged as failed, the later blocks still run
paths.ensure()
CONFIG = 'debug' if DRY else 'base'
HF_CACHE = paths.cache / 'hf' if DRY else Path('/content/hf_cache')
cfg = load_config(CONFIG, [f'model.semantic.cache_dir={HF_CACHE}'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def amp_ctx():   # bf16 on the GPU, as in training
    return runner.autocast(DEVICE)
CK = paths.runs / 'checkpoints'
SUBSETS_FILE = CK / 'eval_subsets.json'
QUALITY_FILE = CK / 'cp1_data_integrity.json'
CACHE = paths.runs / 'cache' / 'tests'
P6_TAR, P6_MAN = paths.runs / 'cache' / 'p6_images.tar', paths.runs / 'cache' / 'p6_images_manifest.json'
BUNDLES = paths.runs / 'explain' / f'p6_unified_ctx_seed{SEED}'
PROBES = paths.runs / 'cp4' / 'probes'
OUT = paths.runs / 'phase_c'
LOCAL = paths.local_data / 'phase_c'
RESULT = OUT / f'phase_c_seed{SEED}.json'
BEST = 'best.pt'
OUT.mkdir(parents=True, exist_ok=True); LOCAL.mkdir(parents=True, exist_ok=True)
def unified_dir(seed):
    return paths.runs / f'eviground_ctx_unified_seed{seed}'
DETECTION_DIR = paths.runs / f'eviground_detection_seed{SEED}'
TARS = ['p1_subset', 'p2', 'p3_brgen', 'p3b', 'p4']
print('commit', info['git_commit'], '| gpu', info.get('gpu_name'), '| device', DEVICE, '| estimate', EST_CU, 'CU')

In [ ]:
# Cell 3: in the dry run, a fake world (tiny checkpoints, test-set tars, P6 cache and bundles); then the pre-flight
if DRY:
    from PIL import Image
    from eviground.eval import p6data
    from eviground.eval.tiled import Item
    from eviground.models.eviground import build_model
    from eviground.train import checkpoint as ckpt
    from eviground.utils.seed import seed_everything
    rng = np.random.default_rng(0)
    seed_everything(42)   # one tiny random model saved as every checkpoint (dry run only)
    _m = build_model(cfg.model)
    _state = {'model': ckpt.trainable_state(_m), 'seed': 42, 'clip_revision': 'tiny',
              'clip_sha256': ckpt.frozen_clip_sha256(_m)}
    for _d in [unified_dir(s) for s in SEEDS] + [DETECTION_DIR]:
        ckpt.save(_d / BEST, _state)
    SRC = paths.local_data / 'dry_src'
    def _png(name, arr):
        p = SRC / name; p.parent.mkdir(parents=True, exist_ok=True); Image.fromarray(arr).save(p); return str(p)
    def _img(h=96, w=128):
        return (rng.random((h, w, 3)) * 255).astype(np.uint8)
    def _mask(h=96, w=128, v=255):
        m = np.zeros((h, w), np.uint8); m[h // 4:h // 2, w // 4:w // 2] = v; return m
    ROWS = {'p1_subset': [], 'p2': [], 'p3_brgen': [], 'p3b': [], 'p4': []}
    for g in ('adm', 'midjourney'):
        for lab in ('real', 'generated'):
            for i in range(2):
                uid = f'genimage_val/{g}/{lab}/{i}.png'
                ROWS['p1_subset'].append({'uid': uid, 'source': 'genimage_val', 'label': lab, 'generator': g,
                                          'role': 'test_P1', 'path': _png(uid, _img())})
    for i in range(3):
        uid = f'diffseg30k/validation/{i}.png'
        ROWS['p2'].append({'uid': uid, 'source': 'diffseg30k', 'label': 'edited', 'role': 'test_P2',
                           'base_type': 'real', 'mask_base': '', 'path': _png(uid, _img()),
                           'mask_path': _png(uid + '.mask.png', _mask(v=2))})
    for i, meth in enumerate(('LaMa', 'SDXL')):
        uid = f'BR-Gen/Forged/{meth}/Stuff/ImageNet/{i}_stuff.png'
        ROWS['p3_brgen'].append({'uid': uid, 'source': 'brgen', 'label': 'edited', 'path': _png(uid, _img()),
                                 'mask_path': _png(f'BR-Gen/Mask/Stuff/ImageNet/{i}_stuff.png', _mask())})
    for lab in ('real', 'full_synthetic', 'tampered'):
        uid = f'sidset/validation/{lab}/0.png'
        r = {'uid': uid, 'source': 'sidset', 'label': lab, 'role': 'test_P3b', 'path': _png(uid, _img())}
        if lab == 'tampered':
            r['mask_path'] = _png(uid + '.mask.png', _mask())
        ROWS['p3b'].append(r)
    for lab in ('real', 'generated'):
        uid = f'Chameleon/test/{lab}/0.png'
        ROWS['p4'].append({'uid': uid, 'source': 'chameleon', 'label': lab, 'path': _png(uid, _img())})
    _build = paths.local_data / 'dry_build'
    for name, rows in ROWS.items():
        testcache.publish(testcache.build(rows, name, _build), _build, CACHE, seed=SEED)
    QUALITY_FILE.parent.mkdir(parents=True, exist_ok=True)
    QUALITY_FILE.write_text(json.dumps({'result': {'quality_distribution_genimage_real': {'table': {'pairs': [[90, '4:2:0', 5]]}}}}))
    items, part_of = [], {}
    DRY_PROBS = {'edited': {'real': 0.05, 'generated': 0.05, 'edited': 0.9},
                 'real': {'real': 0.8, 'generated': 0.15, 'edited': 0.05},
                 'generated': {'real': 0.1, 'generated': 0.85, 'edited': 0.05}}
    for i, part in enumerate(['diffseg_heldout', 'sidset_tampered', 'brgen', 'real', 'generated', 'sidset_tampered']):
        img = _img()
        edited = part in p6data.EDITED_PARTS
        mask = np.zeros((96, 128), bool); mask[20:60, 30:90] = True
        meta = {'part': part, 'label': 'edited' if edited else part}
        if i == 5:   # an aspect-mismatched SID-Set mask (flagged)
            meta.update(mask_resized_from=[60, 128], mask_aspect_mismatch=True)
        uid = f'dry/{part}/{i}.png'
        items.append(Item(uid, img, mask if edited else None, None, meta)); part_of[uid] = part
        probs = DRY_PROBS['edited' if edited else part]
        mp = np.zeros((96, 128)); mp[20:60, 30:90] = 0.9 * edited
        rec, rles = evidence.evidence_record(probs, 0.6, mp, mp + rng.random((96, 128)) * 0.1, img)
        evidence.save_bundle(BUNDLES, uid, rec, rles, {'decoder': mp}, probs, 0.6)
    p6data.write_cache(items, P6_TAR, P6_MAN)
    _p1 = [{k: v for k, v in r.items() if k != 'path'} for r in ROWS['p1_subset']]
    subsets.save({'P6': subsets._record('P6', [{'uid': u} for u in part_of], SEED, 'dry', part_of=part_of),
                  'P5': subsets.p5_subset(_p1, ROWS['p2'], per_label=1, diffseg=1)}, SUBSETS_FILE)
GPU = info.get('gpu_name')
if not DRY:
    assert session.gpu_type(GPU) == 'L4', f'Phase C runs on an L4 (bf16, as in training), found {GPU}'
    assert isinstance(CU_BALANCE_START, (int, float)), 'type the Colab balance into CU_BALANCE_START (cell 2)'
    assert isinstance(CU_CAP, (int, float)), "type Dhanush's cap into CU_CAP (cell 2)"
CAP = phasec.session_cap(CU_CAP, PHASE_C_BALANCE_START, CU_BALANCE_START)   # refuses when nothing is left
assert CAP <= CU_BALANCE_START, 'the cap is above the balance'
print(f'cap this session: {CAP} CU = CU_CAP {CU_CAP} - spent since the Phase C start '
      f'({PHASE_C_BALANCE_START} - {CU_BALANCE_START})')
NEEDED = {f'unified seed {s}': unified_dir(s) / BEST for s in SEEDS}
NEEDED.update({'detection seed 42': DETECTION_DIR / BEST, 'eval_subsets.json': SUBSETS_FILE,
               'CP1 quality distribution': QUALITY_FILE, 'P6 tar': P6_TAR, 'P6 manifest': P6_MAN,
               'evidence bundles (notebook 14)': BUNDLES})
NEEDED.update({f'tar {n}': testcache.paths(CACHE, n)[0] for n in TARS})
PREFLIGHT = session.preflight(
    {'notebook': '17 Phase C', 'gpu': GPU, 'cap (CU)': CU_CAP, 'cap this session (CU)': CAP,
     'Phase C start balance': PHASE_C_BALANCE_START, 'balance at start': CU_BALANCE_START,
     'blocks': list(EST), 'result file': RESULT},
    paths=NEEDED, est_cu=min(EST_CU, CAP), cu_left=CU_BALANCE_START)
for n in TARS:
    assert testcache.is_ready(CACHE, n), f'{n}: no verified tar on Drive (run notebook 16)'
STORED = json.loads(SUBSETS_FILE.read_text())
for key in ('P5', 'P6'):
    assert subsets.intact(STORED[key]), f'{key} in eval_subsets.json was changed: stop and ask Dhanush'
print('P5', STORED['P5']['images'], 'images', STORED['P5']['sha256'][:12], '| P6', STORED['P6']['images'])

In [ ]:
# Cell 4: staging (each tar once, checked member by member), models, probes, predictors
STAGED, STAGE_MIN = {}, {}
def rows_of(name):
    if name not in STAGED:
        t = time.time()
        STAGED[name] = testcache.stage(CACHE, name, LOCAL)
        STAGE_MIN[name] = round((time.time() - t) / 60, 2)
        print(f'staged {name}: {len(STAGED[name])} rows in {STAGE_MIN[name]} min')
    return STAGED[name]

MODELS = {}
def model_of(key):
    '''key: an int seed (unified final configuration) or 'detection'.'''
    if key not in MODELS:
        path = DETECTION_DIR / BEST if key == 'detection' else unified_dir(key) / BEST
        MODELS.clear()   # one model on the GPU at a time
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        MODELS[key] = runner.load_trained(cfg.model, path, DEVICE)
    return MODELS[key]

def probe_of(profile):
    path = PROBES / f'probe_{profile}.pt'
    if not path.is_file():
        print(f'probe {profile}: not cached ({path}); its rows are skipped')
        return None
    return clip_probe.ProbeHead.from_state(torch.load(path, weights_only=True)['head']).to(DEVICE)

def predictor(key, with_probe=False):
    profile = 'detection' if key == 'detection' else 'unified'
    return runner.joint_predictor(model_of(key), probe_of(profile) if with_probe else None, DEVICE)

DIST = harmonize.QualityDistribution.from_dict(
    json.loads(QUALITY_FILE.read_text())['result']['quality_distribution_genimage_real']['table'])

def write_rows(name, records):
    rows = phasec.strip_hist(records)
    if not rows:
        return None
    keys = sorted({k for r in rows for k in r})
    path = OUT / f'{name}_seed{SEED}.csv'
    with open(path, 'w', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=keys)
        w.writeheader()
        w.writerows({k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in r.items()} for r in rows)
    return path.name

# what every block's result depends on: a saved block is reused only if all of it is unchanged
_ckpts = {f'unified_s{s}': unified_dir(s) / BEST for s in SEEDS} | {'detection': DETECTION_DIR / BEST}
PROV = {
    'code': phasec.code_tree(REPO),   # src/, configs/ and this notebook (not docs): a docs-only merge reuses blocks
    'checkpoints': {k: testcache.sha256_file(p) for k, p in _ckpts.items()},
    'probes': {p.name: testcache.sha256_file(p) for p in sorted(PROBES.glob('probe_*.pt'))},
    'tars': {n: json.loads(testcache.paths(CACHE, n)[1].read_text())['tar_sha256'] for n in TARS},
    'p6_tar': json.loads(P6_MAN.read_text()).get('tar_sha256'),
    'subsets': {k: STORED[k]['sha256'] for k in ('P5', 'P6')},
    'p5_levels': perturb.LEVELS,
    'cc_rule': {'delta': cc.DELTA, 'min_token_share': cc.MIN_TOKEN_SHARE, 'n_random': cc.N_RANDOM},
}
print('provenance:', {k: (v if isinstance(v, str) else len(v)) for k, v in PROV.items()})
LOG = {}
def block(name, fn):
    return phasec.run_block(name, EST[name], fn, out_dir=OUT, log=LOG, t0=T0, cu_cap=CAP, rate=RATE,
                            provenance=PROV)

def fetched(tasks):   # items decoded (and harmonized / perturbed) in background threads, in order
    return phasec.prefetch(tasks, **PREFETCH)

In [ ]:
# Cell 5: P1, P2, P3 for each seed (seed 42 first), and the detection model on P1
def p1_block(key, tag, with_probe):
    rows = rows_of('p1_subset')
    gen = [r for r in rows if r['label'] == 'generated']
    names = {'model': tag, 'probe': f'{tag}_probe'}
    predict = predictor(key, with_probe)
    recs, stats = [], {}
    for version, rs in (('raw', rows), ('harmonized', gen)):
        r, stats[version] = runner.p1_pass(predict, fetched(testsets.file_item_tasks(rs, version=version, dist=DIST)), DEVICE, names, version)
        recs += r
    recs += [{**r, 'version': 'harmonized'} for r in recs if r['version'] == 'raw' and r['label'] == 'real']
    out = {'table': cp4.p1_table(recs), 'stats': stats, 'csv': write_rows(f'p1_{tag}', recs)}
    if key != 'detection':
        out['false_edited_real'] = cp4.false_edited(recs, tag)
    return out

def p2_block(seed):
    diag = runner.diagnosis_pass(predictor(seed), fetched(testsets.file_item_tasks(rows_of('p2'))), DEVICE, ceilings=False, sweep=False)
    write_rows(f'p2_unified_s{seed}', [{'uid': r['uid'], **{k: v for k, v in r.items() if k in ('base_type', 'labels')}}
                                       for r in diag['edited']])
    return {'p2': runner.p2_summary(diag['edited'], 'model'), 'stats': diag['stats']}

def p3_block(seed):
    recs, stats = phasec.score(predictor(seed), fetched(phasec.item_tasks(rows_of('p3_brgen'))), DEVICE)
    return {'p3': phasec.p3_summary(recs), 'stats': stats, 'csv': write_rows(f'p3_unified_s{seed}', recs)}

RESULTS = {}
if DRY:   # the crash-safety check: this block raises, is logged as failed, and the session goes on
    RESULTS['dry_fail'] = block('dry_fail', lambda: 1 / 0)
for seed in SEEDS:
    RESULTS[f's{seed}_p1'] = block(f's{seed}_p1', lambda s=seed: p1_block(s, f'unified_s{s}', s == SEED))
    RESULTS[f's{seed}_p2'] = block(f's{seed}_p2', lambda s=seed: p2_block(s))
    RESULTS[f's{seed}_p3'] = block(f's{seed}_p3', lambda s=seed: p3_block(s))
    if seed == SEED:
        RESULTS['det_p1'] = block('det_p1', lambda: p1_block('detection', 'detection', True))
for name, r in RESULTS.items():
    if r and name.endswith('_p1'):
        print(name, {h: {v: round(t['mean_acc'], 4) for v, t in vs.items()} for h, vs in r['table'].items()})
    elif r and name.endswith('_p2'):
        print(name, 'P2 edited IoU', r['p2']['all'].get('iou_edited'))
    elif r and name.endswith('_p3'):
        print(name, 'P3 edited IoU (unseen methods)', r['p3']['unseen_methods'].get('iou_edited'))
print(LOG)

In [ ]:
# Cell 6: counterfactual consistency of every E region of P6 (unified seed 42; rule fixed 24 Nov)
def cc_block():
    P6 = STORED['P6']
    local = paths.local_data / 'p6_cache'
    local_tar = paths.local_data / 'p6_images.tar'
    shutil.copyfile(P6_TAR, local_tar)
    cache = p6cache.read_cache(local_tar, P6_MAN, local)
    model = model_of(SEED)
    regions, same_label = [], []
    for x in p6cache.cached_arrays(P6['uids'], cache, local):
        b = evidence.load_bundle(BUNDLES, x['uid'])
        e_ids = [k for k in b['regions'] if k.startswith('E')]
        if not e_ids:
            continue
        tokens = cc.image_tokens(model, x['image'], amp_ctx, DEVICE)
        flagged = p6cache.is_aspect_mismatched(x['meta'], x['image'].shape[:2])
        for rid in e_ids:
            assert b['regions'][rid].shape == x['image'].shape[:2], (   # a mismatch fails the block
                f"{x['uid']} {rid}: region mask {b['regions'][rid].shape} vs image {x['image'].shape[:2]}")
            r = cc.region_cc(model, tokens, b['regions'][rid], seed=cc.seed_for(x['uid'], rid, SEED), amp=amp_ctx)
            regions.append({'uid': x['uid'], 'region': rid, 'part': x['meta'].get('part'),
                            'label': x['meta'].get('label'), 'flagged_mask': flagged, **r})
        same_label.append(evidence.PRED_KEYS[regions[-1]['pred_label']] == b['evidence']['verdict'])
    keep = [r for r in regions if not r['flagged_mask']]
    parts = sorted({r['part'] for r in regions})
    out = {'all': cc.cc_summary(keep), 'all_incl_flagged': cc.cc_summary(regions),
           'by_part': {p: cc.cc_summary([r for r in keep if r['part'] == p]) for p in parts},
           'edited_images_only': cc.cc_summary([r for r in keep if r['label'] == 'edited']),
           'images_with_regions': len(same_label), 'pred_label_matches_bundle': phasec._mean(same_label),
           'rule': {'delta': cc.DELTA, 'min_token_share': cc.MIN_TOKEN_SHARE, 'n_random': cc.N_RANDOM},
           'csv': write_rows('cc_regions', regions)}
    return out

RESULTS['cc'] = block('cc', cc_block)
if RESULTS['cc']:
    print('CC', RESULTS['cc']['all'], '| incl. flagged', RESULTS['cc']['all_incl_flagged']['cc'])

In [ ]:
# Cell 7: P3b SID-Set validation (unified seed 42) and P4 Chameleon (both models, seed 42)
def p3b_block():
    recs, stats = phasec.score(predictor(SEED), fetched(phasec.item_tasks(rows_of('p3b'))), DEVICE)
    return {'p3b': phasec.p3b_summary(recs), 'stats': stats, 'csv': write_rows('p3b_unified', recs)}

def p4_block():
    out, recs = {}, []
    for key, tag in ((SEED, 'unified'), ('detection', 'detection')):
        r, out[f'stats_{tag}'] = phasec.score(predictor(key), fetched(phasec.item_tasks(rows_of('p4'))), DEVICE,
                                              heads={'model': tag}, masks=False)
        out[tag] = phasec.p4_summary(r, tag)
        recs += r
    out['csv'] = write_rows('p4', recs)
    return out

RESULTS['p3b'] = block('p3b', p3b_block)
RESULTS['p4'] = block('p4', p4_block)
if RESULTS['p3b']:
    print('P3b acc', RESULTS['p3b']['p3b']['acc'], '| macro F1', RESULTS['p3b']['p3b']['macro_f1'])
if RESULTS['p4']:
    print('P4', {k: RESULTS['p4'][k]['acc'] for k in ('unified', 'detection')})

In [ ]:
# Cell 8: P5 robustness, 13 conditions (seed 42): detection on the GenImage part, unified on the DiffSeg30k part
P5_PARTS = STORED['P5']['part_of']
def p5_block(key, tag, tar, part, masks):
    rows = [r for r in rows_of(tar) if P5_PARTS.get(r['uid']) == part]
    assert rows, f'P5 {part}: no rows found in {tar}'
    recs, stats = [], {}
    for cond, kind, level in perturb.iter_conditions():
        r, stats[cond] = phasec.score(predictor(key), fetched(phasec.item_tasks(rows, kind=kind, level=level)), DEVICE,
                                      heads={'model': tag}, masks=masks)
        recs += r
    return {'p5': phasec.p5_summary(recs), 'images': len(rows), 'stats': stats,
            'levels': perturb.LEVELS, 'csv': write_rows(f'p5_{tag}', recs)}

RESULTS['p5_det'] = block('p5_det', lambda: p5_block('detection', 'detection', 'p1_subset', 'genimage', False))
RESULTS['p5_uni'] = block('p5_uni', lambda: p5_block(SEED, 'unified', 'p2', 'diffseg_heldout', True))
for name in ('p5_det', 'p5_uni'):
    if RESULTS[name]:
        for head, conds in RESULTS[name]['p5'].items():
            print(name, head, {c: round(next(iter(v.values())).get('acc', next(iter(v.values())).get('iou_edited') or 0), 3)
                               for c, v in conds.items()})

In [ ]:
# Cell 9: efficiency (seed 42 unified and the detection model): parameters, tiles/s, images/s, GPU memory
def tiles_per_s(model):
    x = torch.rand(8 if DRY else 32, 3, 256, 256, device=DEVICE)
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    n = 3 if DRY else 30
    with torch.no_grad(), amp_ctx():
        for _ in range(2 if DRY else 5):
            model(x)
        sync()
        t = time.time()
        for _ in range(n):
            model(x)
        sync()
    return n * x.shape[0] / (time.time() - t), x.shape[0]

def peak_gb():
    return round(torch.cuda.max_memory_allocated() / 1e9, 3) if torch.cuda.is_available() else None

def timings_block():
    out = {}
    rows = rows_of('p1_subset')[: 8 if DRY else 200]
    for key, tag in ((SEED, 'unified'), ('detection', 'detection')):
        model = model_of(key)
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        tps, batch = tiles_per_s(model)
        _, st = phasec.score(predictor(key), testsets.file_items(rows), DEVICE, masks=False)
        out[tag] = {**phasec.param_counts(model), 'tiles_per_s_batch': round(tps, 1), 'batch': batch,
                    'images_per_s_p1': round(st['images'] / max(st['seconds'], 1e-6), 2), 'p1_images': st['images'],
                    'peak_gpu_memory_gb': peak_gb()}
    return out

RESULTS['timings'] = block('timings', timings_block)
print(RESULTS['timings'])

In [ ]:
# Cell 10: the result file, the compute record (balance), then release the runtime
RESULT_DATA = {
    'step': 'Phase C test-set evaluation', 'dry_run': DRY, 'branch': BRANCH, 'seed': SEED, 'seeds': SEEDS,
    'session': info, 'git_commit': info['git_commit'], 'cap_cu': CU_CAP, 'cap_cu_session': CAP, 'phase_c_balance_start': PHASE_C_BALANCE_START, 'prefetch': PREFETCH, 'rate_cu_per_h': RATE, 'estimates': EST, 'blocks': LOG.get('blocks', {}),
    'stopped_at_cap': bool(LOG.get('stopped')), 'staging_minutes': STAGE_MIN, 'provenance': PROV,
    'subsets': {k: {'images': STORED[k]['images'], 'sha256': STORED[k]['sha256']} for k in ('P5', 'P6')},
    'results': {k: v for k, v in RESULTS.items() if v is not None},
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
}
RESULT.write_text(json.dumps(RESULT_DATA, indent=1, default=phasec._json_default))
print('saved', RESULT, '| blocks', {k: (v['status'], v.get('minutes'), v.get('images_per_s')) for k, v in LOG.get('blocks', {}).items()})
for k, v in LOG.get('blocks', {}).items():
    if v['status'] == 'failed':
        print(f'FAILED {k}:', v['error'])
COMPUTE = session.finish(OUT, T0, gpu=GPU or 'cpu', name='compute_phase_c.json', balance_start=CU_BALANCE_START,
                         extra={'git_commit': info['git_commit'], 'est_cu': EST_CU, 'cap_cu': CU_CAP, 'cap_cu_session': CAP})
if DRY:
    assert LOG['blocks']['dry_fail']['status'] == 'failed', LOG['blocks']['dry_fail']
    assert 'ZeroDivisionError' in LOG['blocks']['dry_fail']['error']
    assert all(v['status'] == 'done' for k, v in LOG['blocks'].items() if k != 'dry_fail'), LOG
    assert all('images_per_s' in v for k, v in LOG['blocks'].items() if k != 'dry_fail'), LOG   # dry blocks take ~0 s
    assert RESULTS['s42_p1']['_block']['images'] > 0
    assert RESULTS['p5_det']['_block']['images'] > 0
    assert phasec.session_cap(9.7, 23.43, 19.0) == 5.27   # a resumed session: 9.7 - (23.43 - 19.0)
    _again = {}   # a rerun with the same inputs reuses every block; another input reruns it
    assert phasec.run_block('cc', 0.0, lambda: {}, out_dir=OUT, log=_again, t0=T0, cu_cap=CU_CAP, provenance=PROV)
    assert _again['blocks']['cc']['status'] == 'reused'
    phasec.run_block('cc', 0.0, lambda: {'x': 1}, out_dir=OUT, log=_again, t0=T0, cu_cap=CU_CAP,
                     provenance={**PROV, 'code': {'src': 'other'}})
    assert _again['blocks']['cc']['status'] == 'rerun', _again
    assert set(LOG['blocks']) == set(EST)
    assert RESULTS['cc']['all']['regions'] >= 1 and RESULTS['cc']['all_incl_flagged']['regions'] > RESULTS['cc']['all']['regions']
    assert len(RESULTS['p5_det']['p5']['detection']) == len(perturb.conditions())
    assert RESULTS['s42_p3']['p3']['sdxl']['images'] == 1
RELEASED = session.release(DRY)
if DRY:
    assert RELEASED == 'dry', RELEASED